# Data Ingest & Deterministic Splits (IMDb, Local CSV)

**Objective.** 

Load the Kaggle IMDb dataset from `data/raw/IMDB Dataset.csv`, remove exact duplicates on `(text, label)`, assign deterministic IDs (`doc_id = md5(text|label)`), and produce **stratified 80/10/10** splits with a fixed seed so all later experiments are reproducible.

**Why this matters.**
- A clean, deterministic split is the foundation for fair model comparison.
- `doc_id` lets us track documents across preprocessing, ML, and DL notebooks.
- Stratification prevents class imbalance from skewing metrics.

**Outputs**
- `data/processed/train.csv`, `val.csv`, `test.csv` (columns: `doc_id,text,label`)
- `reports/tables/split_stats.json` (class balance per split)
- `reports/tables/dedup_stats.json` (duplicates removed)


## 1) Imports & Directory Setup

The small bootstrap snippet ensures this notebook can import modules from `src/` whether it is run from the `notebooks/` folder or the repo root.


In [4]:
from pathlib import Path
import pandas as pd, json

import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import SEED, DATA_RAW, DATA_PROCESSED, REPORTS_TAB
from src import data_io


## 2) Data Cleanup

This cell performs the full pipeline via reusable functions in `src/data_io.py`:

1. **Load** the local CSV and standardize to `text` (str) and `label` (0/1).
2. **Deduplicate** exact `(text,label)` pairs and create `doc_id = md5(text|label)` (deterministic IDs).
3. **Stratified 80/10/10** split with `SEED=42` (train/val/test) to keep class proportions aligned.
4. **Persist** the three CSVs plus JSON stats for later verification and reporting.



In [5]:
train, val, test = data_io.ingest_and_split()
print("✅ Ingest & split complete.")
print("Saved CSVs →", DATA_PROCESSED)
print("Stats JSON →", REPORTS_TAB / "split_stats.json")


✅ Ingest & split complete.
Saved CSVs → /Users/testsolutions/Documents/school/year3/term2/formative-2-sentiment-analysis/data/processed
Stats JSON → /Users/testsolutions/Documents/school/year3/term2/formative-2-sentiment-analysis/reports/tables/split_stats.json


## 3) Acceptance Gate ✅

We enforce hard checks to ensure the data contract and split integrity:

- **Column contract:** each split must be exactly `["doc_id","text","label"]`.
- **Types:** `label` is integer; `doc_id` is string.
- **No overlaps:** `doc_id` sets across train/val/test are pairwise disjoint.
- **Stratification tolerance:** positive class percentage differs by at most **±0.5%** across splits.



In [ ]:
train = pd.read_csv(DATA_PROCESSED / "train.csv")
val   = pd.read_csv(DATA_PROCESSED / "val.csv")
test  = pd.read_csv(DATA_PROCESSED / "test.csv")

with open(REPORTS_TAB / "split_stats.json") as f:
    split_stats = json.load(f)

# Verify column contract
assert list(train.columns) == ["doc_id","text","label"]
assert list(val.columns)   == ["doc_id","text","label"]
assert list(test.columns)  == ["doc_id","text","label"]

# Verify types
for df in (train, val, test):
    assert df["label"].dtype.kind in "iu", "label must be integer"
    assert df["doc_id"].dtype == "object", "doc_id must be string"

# Verify there are no overlaps
assert set(train.doc_id).isdisjoint(set(val.doc_id))
assert set(train.doc_id).isdisjoint(set(test.doc_id))
assert set(val.doc_id).isdisjoint(set(test.doc_id))

# Verify splits sizes
base_pos = split_stats["train"]["pos_pct"]
for k in ["val","test"]:
    assert abs(split_stats[k]["pos_pct"] - base_pos) <= 0.5

print("✅ Acceptance Gate passed.")
split_stats


✅ Acceptance Gate passed.


{'train': {'total': 39665,
  'pos': 19907,
  'neg': 19758,
  'pos_pct': 50.19,
  'neg_pct': 49.81},
 'val': {'total': 4958,
  'pos': 2488,
  'neg': 2470,
  'pos_pct': 50.18,
  'neg_pct': 49.82},
 'test': {'total': 4959,
  'pos': 2489,
  'neg': 2470,
  'pos_pct': 50.19,
  'neg_pct': 49.81}}

## 4) Next Steps

Proceed to `01_Text_Cleaning_and_EDA.ipynb` to apply the deterministic cleaning pipeline and produce EDA figures that guide feature and model choices.
